# Colab T4 fine-tune -> Hugging Face (push only if val loss improves)

**Run from local VS Code:** open this file, click *Select Kernel -> Colab -> New Colab Server -> T4 GPU*, then *Run All*.

You need a Hugging Face token with **write** access, saved in Colab Secrets as `HF_TOKEN` (otherwise you will be prompted to paste it).

Before running, change `HF_USER` in the config cell.

In [2]:
%pip -q install 'transformers>=4.46,<5' 'peft>=0.13' 'datasets>=3.0' 'accelerate>=1.0' huggingface_hub

In [3]:
!nvidia-smi


/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: select the Colab kernel with a T4 runtime'
print(torch.cuda.get_device_name(0))

In [ ]:
import os, json

HF_USER = 'jeet390'   # <-- change me
MODEL_REPO = f'{HF_USER}/chat-model-lora'
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
DATASET = 'trl-lib/Capybara'   # any dataset with a 'messages' column and train/test splits
N_TRAIN, N_EVAL, MAX_LEN = 4000, 300, 512
OUT_DIR, ADAPTER_DIR = 'out', 'adapter'

In [ ]:
import getpass
from huggingface_hub import login, HfApi

try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = os.environ.get('HF_TOKEN') or getpass.getpass('HF token (write access): ')

login(token)
api = HfApi()
api.create_repo(MODEL_REPO, private=True, exist_ok=True)
api.create_branch(MODEL_REPO, branch='checkpoints', exist_ok=True)  # resume points live here, not on main
print('Repo ready:', MODEL_REPO)

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

train_ds = load_dataset(DATASET, split=f'train[:{N_TRAIN}]')
eval_ds = load_dataset(DATASET, split=f'test[:{N_EVAL}]')

def tokenize(ex):
    text = tok.apply_chat_template(ex['messages'], tokenize=False)
    return tok(text, truncation=True, max_length=MAX_LEN)

train_ds = train_ds.map(tokenize, remove_columns=train_ds.column_names)
eval_ds = eval_ds.map(tokenize, remove_columns=eval_ds.column_names)
print(len(train_ds), 'train /', len(eval_ds), 'eval examples')

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

# fp32 base weights + fp16 autocast: T4 has no bf16, and LoRA params must be fp32 for fp16 training
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32).to('cuda')
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
import glob
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling, TrainerCallback

class CheckpointToHub(TrainerCallback):
    '''Upload each checkpoint to the `checkpoints` branch so a dropped Colab session can resume.'''
    def on_save(self, args, state, control, **kw):
        ckpt = f'{args.output_dir}/checkpoint-{state.global_step}'
        try:
            api.upload_folder(folder_path=ckpt, repo_id=MODEL_REPO, revision='checkpoints',
                              path_in_repo='latest', commit_message=f'checkpoint step {state.global_step}')
        except Exception as e:
            print('checkpoint upload failed (training continues):', e)

args = TrainingArguments(
    output_dir=OUT_DIR, num_train_epochs=1,
    per_device_train_batch_size=4, gradient_accumulation_steps=4, per_device_eval_batch_size=4,
    learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.03,
    fp16=True, logging_steps=10,
    eval_strategy='steps', eval_steps=50, save_strategy='steps', save_steps=50, save_total_limit=2,
    report_to='none',
)
trainer = Trainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
    data_collator=DataCollatorForLanguageModeling(tok, mlm=False),
    callbacks=[CheckpointToHub()],
)

# Resume if this VM still has checkpoints from an interrupted run in the same session
resume = bool(glob.glob(f'{OUT_DIR}/checkpoint-*'))
trainer.train(resume_from_checkpoint=resume)

In [ ]:
from huggingface_hub import hf_hub_download

new_loss = trainer.evaluate()['eval_loss']

try:
    prev_loss = json.load(open(hf_hub_download(MODEL_REPO, 'metrics.json')))['eval_loss']
except Exception:
    prev_loss = float('inf')   # first run: nothing published yet

print(f'new eval_loss={new_loss:.4f} | published eval_loss={prev_loss:.4f}')

if new_loss < prev_loss:
    version = f'v1.{len(api.list_repo_refs(MODEL_REPO).tags) + 1}'
    model.save_pretrained(ADAPTER_DIR)
    tok.save_pretrained(ADAPTER_DIR)
    with open(f'{ADAPTER_DIR}/metrics.json', 'w') as f:
        json.dump({'eval_loss': new_loss, 'version': version, 'base_model': BASE_MODEL}, f)
    api.upload_folder(folder_path=ADAPTER_DIR, repo_id=MODEL_REPO,
                      commit_message=f'{version}: eval_loss {new_loss:.4f}')
    api.create_tag(MODEL_REPO, tag=version)
    print(f'Pushed {version} to https://huggingface.co/{MODEL_REPO}')
else:
    print('No improvement - nothing pushed, so the webhook and deployment will not fire.')

In [ ]:
# Quick sanity check on the GPU
model.eval()
msgs = [{'role': 'user', 'content': 'Give me two tips for learning Python.'}]
prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tok(prompt, return_tensors='pt').to('cuda')
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
print(tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))